In [1]:
import json
import logging
from pathlib import Path
from typing import Any, Dict

import requests

In [2]:
def save_json(obj, path: str):
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    with p.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
    return str(p)


def load_json(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def load_connector_json(file_path: str) -> Dict[str, Any]:
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def create_connector(url: str, config: dict):
    r = requests.post(f"{url}/connectors", headers={"Content-Type": "application/json"}, data=json.dumps(config), timeout=10)
    logging.info(f"status code: {r.status_code}")
    return r.text


def status_connector(url: str, name: str):
    r = requests.get(f"{url}/connectors/{name}/status", timeout=10)
    logging.info(f"status code: {r.status_code}")
    return r.json() if r.headers.get("Content-Type", "").startswith("application/json") else r.text


def delete_connector(url: str, name: str):
    r = requests.delete(f"{url}/connectors/{name}", timeout=10)
    logging.info(f"status code: {r.status_code}")
    return r


def upsert_connector(url: str, config: dict):
    name = config["name"]
    r = requests.get(f"{url}/connectors/{name}", timeout=10)
    if r.status_code == 200:
        r2 = requests.put(f"{url}/connectors/{name}/config", headers={"Content-Type": "application/json"}, data=json.dumps(config), timeout=10)
        return r2.status_code, r2.text
    elif r.status_code == 404:
        return create_connector(url, config)
    else:
        return r.status_code, r.text

In [3]:
_kafka_connect_url = "http://confluent-kafka-connect.mmix.io:8083"

## Debezium Mysql External CDC Connector (Source)

In [6]:
mysql_external_source_config = load_json("./config/mmix-mysql-external-source-connector.json")
create_connector(_kafka_connect_url, mysql_external_source_config)
#status_connector(_kafka_connect_url, mysql_external_source_config["name"])
#delete_connector(_kafka_connect_url, mysql_external_source_config["name"])

'{"name":"mmix-mysql-external-source-connector","config":{"connector.class":"io.debezium.connector.mysql.MySqlConnector","tasks.max":"1","database.hostname":"mysql-primary.mmix.io","database.port":"3306","database.user":"debezium","database.password":"debezium","database.server.id":"1","database.server.name":"mmix-mysql-external-inventory","database.include.list":"external","snapshot.mode":"initial","topic.prefix":"mmix.mysql.external.source","transforms":"unwrap","transforms.unwrap.type":"io.debezium.transforms.ExtractNewRecordState","transforms.unwrap.drop.tombstones":"true","transforms.unwrap.delete.handling.mode":"rewrite","schema.history.internal.kafka.bootstrap.servers":"confluent-kafka-broker.mmix.io:9093","schema.history.internal.kafka.topic":"debezium.mysql.source.schema.changes.inventory","schema.history.internal.producer.security.protocol":"SASL_PLAINTEXT","schema.history.internal.producer.sasl.mechanism":"PLAIN","schema.history.internal.producer.sasl.jaas.config":"org.apach

## Debezium Mysql MMIX CDC Connector (Source)

In [7]:
mysql_mmix_source_config = load_json("./config/mmix-mysql-mmix-source-connector.json")
create_connector(_kafka_connect_url, mysql_mmix_source_config)
#status_connector(_kafka_connect_url, mysql_mmix_source_config["name"])
#delete_connector(_kafka_connect_url, mysql_mmix_source_config["name"])

'{"name":"mmix-mysql-mmix-source-connector","config":{"connector.class":"io.debezium.connector.mysql.MySqlConnector","tasks.max":"1","database.hostname":"mysql-primary.mmix.io","database.port":"3306","database.user":"debezium","database.password":"debezium","database.server.id":"2","database.server.name":"mmix-mysql-mmix-inventory","database.include.list":"mmix","snapshot.mode":"initial","topic.prefix":"mmix.mysql.mmix.source","transforms":"unwrap","transforms.unwrap.type":"io.debezium.transforms.ExtractNewRecordState","transforms.unwrap.drop.tombstones":"true","transforms.unwrap.delete.handling.mode":"rewrite","schema.history.internal.kafka.bootstrap.servers":"confluent-kafka-broker.mmix.io:9093","schema.history.internal.kafka.topic":"debezium.mysql.source.schema.changes.inventory","schema.history.internal.producer.security.protocol":"SASL_PLAINTEXT","schema.history.internal.producer.sasl.mechanism":"PLAIN","schema.history.internal.producer.sasl.jaas.config":"org.apache.kafka.common.s

## Kafka Postgresql External JDBC CDC Connector (Sync)

In [8]:
postgresql_external_sync_config = load_json("./config/mmix-postgresql-external-sync-connector.json")
create_connector(_kafka_connect_url, postgresql_external_sync_config)
#status_connector(_kafka_connect_url, postgresql_external_sync_config["name"])
#delete_connector(_kafka_connect_url, postgresql_external_sync_config["name"])

'{"name":"mmix-postgresql-external-sync-connector","config":{"connector.class":"io.confluent.connect.jdbc.JdbcSinkConnector","tasks.max":"1","topics.regex":"^mmix\\\\.mysql\\\\.external\\\\.source\\\\..*$","connection.url":"jdbc:postgresql://postgresql.mmix.io:5432/warehouse","connection.user":"warehouse","connection.password":"warehouse","auto.create":"true","auto.evolve":"true","insert.mode":"upsert","pk.mode":"record_key","delete.enabled":"true","key.converter":"io.confluent.connect.avro.AvroConverter","value.converter":"io.confluent.connect.avro.AvroConverter","key.converter.schema.registry.url":"http://confluent-schema-registry.mmix.io:8081","value.converter.schema.registry.url":"http://confluent-schema-registry.mmix.io:8081","transforms":"route","transforms.route.type":"org.apache.kafka.connect.transforms.RegexRouter","transforms.route.regex":"mmix\\\\.mysql\\\\.external\\\\.source\\\\.external\\\\.(.*)","transforms.route.replacement":"$1","table.name.format":"external.${topic}",

## Kafka Postgresql MMIX JDBC CDC Connector (Sync)

In [9]:
postgresql_mmix_sync_config = load_json("./config/mmix-postgresql-mmix-sync-connector.json")
create_connector(_kafka_connect_url, postgresql_mmix_sync_config)
#status_connector(_kafka_connect_url, postgresql_mmix_sync_config["name"])
#delete_connector(_kafka_connect_url, postgresql_mmix_sync_config["name"])

'{"name":"mmix-postgresql-mmix-sync-connector","config":{"connector.class":"io.confluent.connect.jdbc.JdbcSinkConnector","tasks.max":"1","topics.regex":"^mmix\\\\.mysql\\\\.mmix\\\\.source\\\\..*$","connection.url":"jdbc:postgresql://postgresql.mmix.io:5432/warehouse","connection.user":"warehouse","connection.password":"warehouse","auto.create":"true","auto.evolve":"true","insert.mode":"upsert","pk.mode":"record_key","delete.enabled":"true","key.converter":"io.confluent.connect.avro.AvroConverter","value.converter":"io.confluent.connect.avro.AvroConverter","key.converter.schema.registry.url":"http://confluent-schema-registry.mmix.io:8081","value.converter.schema.registry.url":"http://confluent-schema-registry.mmix.io:8081","transforms":"route","transforms.route.type":"org.apache.kafka.connect.transforms.RegexRouter","transforms.route.regex":"mmix\\\\.mysql\\\\.mmix\\\\.source\\\\.mmix\\\\.(.*)","transforms.route.replacement":"$1","table.name.format":"mmix.${topic}","errors.tolerance":"

## Confluent Kafka Connector S3 등록 (Sync)

In [11]:
s3_config = load_json("./config/mmix-s3-mmix-sync-connector.json")
create_connector(_kafka_connect_url, s3_config)
# status_connector(_kafka_connect_url, s3_config["name"])
# delete_connector(_kafka_connect_url, s3_config["name"])

'{"name":"mmix-s3-mmix-sync-connector","config":{"tasks.max":"1","connector.class":"io.confluent.connect.s3.S3SinkConnector","flush.size":"10000","rotate.interval.ms":"600000","partition.duration.ms":"3600000","locale":"ko_KR","timezone":"UTC","topics.dir":"messaging/mmix/mmix","topics.regex":"^mmix\\\\.mysql\\\\.mmix\\\\.source\\\\..*$","partitioner.class":"io.confluent.connect.storage.partitioner.TimeBasedPartitioner","format.class":"io.confluent.connect.s3.format.parquet.ParquetFormat","parquet.codec":"snappy","key.converter":"org.apache.kafka.connect.storage.StringConverter","value.converter":"io.confluent.connect.avro.AvroConverter","value.converter.schema.registry.url":"http://confluent-schema-registry.mmix.io:8081","value.converter.schemas.enable":"true","path.format":"\'year\'=YYYY/\'month\'=MM/\'day\'=dd/\'hour\'=HH","storage.class":"io.confluent.connect.s3.storage.S3Storage","store.url":"http://minio.mmix.io:9000","aws.access.key.id":"mmix","aws.secret.access.key":"mmixmmix",